# Typhoon and Flood Control Infrastructure Dataset Integration

This notebook integrates the **cleaned typhoon dataset (2019–2025)** with the **flood control infrastructure projects dataset** to create a unified dataset for analysis. The objective is to examine how previously completed flood control infrastructure may relate to the impact of typhoon events across different provinces.

To achieve this, the notebook first loads the cleaned typhoon observations and infrastructure project records. Date fields are standardized to ensure accurate temporal comparisons between typhoon occurrences and infrastructure completion dates. This allows the analysis to determine which flood control projects were already completed before a given typhoon event occurred.

For each typhoon observation, the notebook calculates several cumulative infrastructure metrics within the same province. These include the **total flood control budget completed prior to the typhoon**, the **cumulative budget variance (difference between approved and contracted costs)**, and a **variance ratio** representing the proportion of budget variance relative to total spending. These indicators provide contextual information about the level of flood control investment and financial efficiency in each province before the storm.

The resulting dataset combines **meteorological observations with infrastructure investment data**, enabling further analysis on the relationship between **typhoon intensity, rainfall impacts, and the presence of completed flood control projects**. The final merged dataset is exported for use in subsequent analytical and modeling steps.

## Import
Import **numpy**, **pandas**, and **seaborn** and **matplotlib.pyplot**.

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

### Loading the Cleaned Datasets

In this step, the notebook loads the two cleaned datasets that will be used for the integration process: the **typhoon observations dataset** and the **flood control infrastructure projects dataset**.

The typhoon dataset (`cleaned_2019-2025.csv`) contains weather station observations associated with typhoon events, including rainfall intensity, wind gust measurements, and geographic information. The infrastructure dataset (`cleaned_infra_projects.csv`) contains records of flood control projects, including their locations, budgets, contract costs, and completion dates.

Both files are read using the `pd.read_csv()` function and stored in separate pandas DataFrames: **df_typhoon** and **df_infra**. These DataFrames will serve as the primary data sources for the subsequent steps, where the infrastructure data will be matched with typhoon events based on **province** and **project completion timing**.

In [ ]:
# 1. Load the clean datasets
df_typhoon = pd.read_csv('./data/typhoon-info/cleaned_2019-2025.csv')
df_infra = pd.read_csv('./data/infra-projects/cleaned_infra_projects.csv')

### Standardizing Date Fields

In this step, the date columns in both datasets are converted into a standardized **datetime format**. Proper date formatting is essential for performing accurate time-based comparisons between typhoon events and infrastructure project completion dates.

The **Date** column in the typhoon dataset and the **ActualCompletionDate** column in the infrastructure dataset are converted using the `pd.to_datetime()` function. This ensures that both columns are recognized as datetime objects rather than plain text. Standardizing the date format allows the notebook to correctly determine whether a flood control project was **completed before a typhoon occurred**. 

In [ ]:
# 2. Standardize Dates
df_typhoon['Date'] = pd.to_datetime(df_typhoon['Date'])
df_infra['ActualCompletionDate'] = pd.to_datetime(df_infra['ActualCompletionDate'])

### Computing Cumulative Flood Control Infrastructure Context

In this step, a custom function `get_full_infra_context()` is defined to calculate the amount of flood control infrastructure investment that was already in place when a given typhoon event occurred. The function evaluates each typhoon observation individually and retrieves the relevant infrastructure context based on its **province** and **date**.

For every typhoon record, the function first filters the infrastructure dataset to include only projects that meet two conditions: they must belong to the **same province** as the typhoon observation, and they must have been **completed before the typhoon date**. This ensures that only infrastructure that was already available at the time of the event is included in the calculation.

If no matching infrastructure projects are found, the function returns zero values for all three output metrics. Otherwise, it computes the **cumulative budget completed to date**, the **cumulative budget variance to date**, and the **variance ratio to date**. The cumulative budget represents the total value of completed flood control projects before the event, while the cumulative variance measures the total difference between approved and contracted costs. The variance ratio expresses this variance relative to the total completed budget, providing an indicator of overall cost efficiency.

By returning these values as a pandas Series, the function makes it possible to attach multiple infrastructure-related metrics to each typhoon observation. This creates a richer dataset that reflects not only meteorological conditions, but also the level of flood control investment already present before each storm event.

In [ ]:
# 3. We calculate the flood control spending so far with a given typhoon event
def get_full_infra_context(row):
    # Match Province and ensure the project was finished before the typhoon hit
    mask = (df_infra['Province'] == row['Province']) & (df_infra['ActualCompletionDate'] < row['Date'])
    matching = df_infra[mask]

    if matching.empty:
        return pd.Series([0.0, 0.0, 0.0], 
                         index=['Cumulative_Budget_To_Date', 'Cumulative_Variance_To_Date', 'Variance_Ratio_To_Date'])

    # a. Sum the Budget
    budget_so_far = matching['Final_Budget'].sum()

    # b. Sum the Variance
    variance_so_far = matching['Budget_Variance'].sum()

    # c. Calculate the Ratio
    variance_ratio = (variance_so_far / budget_so_far) if budget_so_far != 0 else 0

    return pd.Series([budget_so_far, variance_so_far, variance_ratio], 
                     index=['Cumulative_Budget_To_Date', 'Cumulative_Variance_To_Date', 'Variance_Ratio_To_Date'])

### Attaching Infrastructure Context to Typhoon Observations

In this step, the infrastructure context calculated by the `get_full_infra_context()` function is applied to each typhoon observation in the dataset. The goal is to enrich the typhoon dataset with information about the **cumulative flood control infrastructure that existed before each storm event**.

The `.apply()` method is used with `axis=1`, which instructs pandas to run the function on **each row** of the typhoon dataset. For every typhoon record, the function evaluates the province and event date, then returns three calculated metrics: cumulative infrastructure budget, cumulative variance, and the variance ratio.

Because the function returns multiple values as a pandas Series, the result of `.apply()` is a new DataFrame (`infra_cols`) containing these calculated columns for every row. The `pd.concat()` function is then used to attach these new columns to the original typhoon dataset along the column axis. By the end of this step, the typhoon dataset is augmented with infrastructure-related metrics that represent the **flood control investment context available at the time each typhoon occurred**.

In [ ]:
# 4. Attach the data
# We use .apply() to create multiple columns at once
infra_cols = df_typhoon.apply(get_full_infra_context, axis=1)
df_typhoon = pd.concat([df_typhoon, infra_cols], axis=1)

### Sorting the Dataset by Event Date

In this step, the typhoon dataset is sorted chronologically based on the **Date** column. Organizing the data in ascending order ensures that typhoon observations are arranged from the earliest event to the most recent.

In [ ]:
# 5. Sort by Date
df_typhoon = df_typhoon.sort_values(by='Date', ascending=True)

### Saving the Final Merged Dataset

After enriching the typhoon dataset with the calculated infrastructure context metrics, the final integrated dataset is exported to a CSV file.

In [ ]:
# 6. Save the final file
df_typhoon.to_csv('./data/merged/typhoon-info-infra-project.csv', index=False)